# Scale MSOA results to LSOA level

Simple division of the total admissions by MSOA by the number of LSOA in that MSOA.

## Code setup

In [1]:
import os
import pandas as pd

## Load data

LSOA to MSOA lookup:

In [2]:
p = os.path.join('data', 'lsoa_to_msoa.csv')
df_lookup = pd.read_csv(p)

In [4]:
df_lookup.head(2)

,lsoa11cd,msoa11cd,ladcd,lsoa11nm,msoa11nm,country
0,E01000001,E02000001,E09000001,City of London 001A,City of London 001,E
1,E01000002,E02000001,E09000001,City of London 001B,City of London 001,E


MSOA-level projections:

In [5]:
p = os.path.join('outputs', 'msoa_stroke_predictions_2035.csv')
df_proj = pd.read_csv(p)

In [6]:
df_proj.head(2)

,MSOA,MSOA11CD,country,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max,admissions_predicted,std
0,Adur 001,E02006534,E,6736,658,626,500,838,0.4,0.6,18.8968,3.756584
1,Adur 002,E02006535,E,5242,512,487,389,652,0.8,1.0,12.6876,2.949388


## Split admissions across LSOA

How many LSOA are in each MSOA?

In [8]:
df_lookup['count'] = 1
s_msoa_count = df_lookup[['msoa11cd', 'count']].groupby('msoa11cd').sum()

In [9]:
s_msoa_count.head(2)

,count
msoa11cd,
E02000001,6
E02000002,4


In [11]:
df_proj_msoa = pd.merge(
    s_msoa_count.reset_index(),
    df_proj[['MSOA11CD', 'admissions_predicted']],
    left_on='msoa11cd', right_on='MSOA11CD', how='right'
)

In [15]:
df_proj_msoa['lsoa_admissions_predicted'] = (
    df_proj_msoa['admissions_predicted'] / df_proj_msoa['count']).round(4)

In [16]:
df_proj_msoa.head(2)

,msoa11cd,count,MSOA11CD,admissions_predicted,lsoa_admissions_predicted
0,E02006534,6,E02006534,18.8968,3.1495
1,E02006535,5,E02006535,12.6876,2.5375


## Make lookup for LSOA to predicted admissions

In [19]:
df_lookup = pd.merge(
    df_lookup,
    df_proj_msoa[['msoa11cd', 'lsoa_admissions_predicted']],
    on='msoa11cd', how='right'
)

In [21]:
df_lookup = df_lookup.drop('count', axis='columns')

In [22]:
df_lookup.head(3)

,lsoa11cd,msoa11cd,ladcd,lsoa11nm,msoa11nm,country,lsoa_admissions_predicted
0,E01031349,E02006534,E07000223,Adur 001A,Adur 001,E,3.1495
1,E01031350,E02006534,E07000223,Adur 001B,Adur 001,E,3.1495
2,E01031351,E02006534,E07000223,Adur 001C,Adur 001,E,3.1495


In [23]:
df_lookup.tail(3)

,lsoa11cd,msoa11cd,ladcd,lsoa11nm,msoa11nm,country,lsoa_admissions_predicted
32840,E01013423,E02002795,E06000014,York 024D,York 024,E,2.1855
32841,E01013424,E02002795,E06000014,York 024E,York 024,E,2.1855
32842,E01013425,E02002795,E06000014,York 024F,York 024,E,2.1855


## Save results

In [24]:
p = os.path.join('outputs', 'lsoa_stroke_predictions_2035.csv')
df_lookup.to_csv(p, index=False)